# 📊 Módulo 08 - Notebook 04: Heatmaps de Correlación y Riesgo

## 🔥 Matrices de Calor para Análisis Multivariado

**Libro:** Saliendo de lo Pandito  
**Módulo:** 08 - Visualización Plotly Databricks Dashboards  
**Duración estimada:** 65 minutos  
**Dificultad:** 🟠 Intermedio-Avanzado  
**Plataforma:** Databricks Free Edition

---

## 🎯 Objetivos de aprendizaje

Al finalizar este notebook serás capaz de:

✅ **Crear** heatmaps (matrices de calor)  
✅ **Visualizar** matrices de correlación  
✅ **Construir** matrices de riesgo  
✅ **Interpretar** patrones en datos multivariados  
✅ **Personalizar** escalas de color

---

## 📋 Pre-requisitos

* ✅ Notebooks 08_01, 08_02, 08_03 completados
* ✅ Conocimiento de correlación
* ✅ Familiaridad con análisis de riesgo

---

## 📚 Contenido

1. Teoría de Heatmaps
2. Matriz de Correlación
3. Matriz de Riesgo
4. Escalas de Color
5. Caso Integrador: Dashboard de Riesgo

---

## 💡 Por qué importa

**Heatmaps revelan patrones ocultos:**

* 🔍 **Correlaciones:** Detectar relaciones entre variables
* ⚠️ **Riesgo:** Visualizar matriz probabilidad-impacto
* 📊 **Performance:** Identificar clusters de alto rendimiento

**Un vistazo rápido = Detectar patrones en segundos**

In [0]:
import pandas as pd
import numpy as np

print("💾 CARGANDO DATOS REALES DESDE UNITY CATALOG")
print("="*70)

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    # Cargar tabla de ventas de Los Andes Market
    df_ventas = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3").toPandas()
    df_ventas['fecha'] = pd.to_datetime(df_ventas['fecha'])
    df_ventas['mes'] = df_ventas['fecha'].dt.month
    df_ventas['año'] = df_ventas['fecha'].dt.year
    
    # Crear estructura para correlaciones
    # Pivot para tener ventas por sucursal en columnas
    df_pivot = df_ventas.pivot_table(
        values='ventas',
        index='fecha',
        columns='sucursal_nombre',
        aggfunc='sum'
    ).fillna(0)
    
    # Calcular matriz de correlación
    corr_matrix = df_pivot.corr()
    
    print(f"\n✅ Datos reales cargados exitosamente")
    print(f"   📊 Registros: {len(df_ventas):,}")
    print(f"   📅 Período: {df_ventas['fecha'].min().strftime('%Y-%m-%d')} a {df_ventas['fecha'].max().strftime('%Y-%m-%d')}")
    print(f"   🏪 Sucursales: {df_ventas['sucursal_id'].nunique()}")
    print(f"   📍 Ubicación: Mendoza, Argentina")
    
    print(f"\n📊 Matriz de correlación calculada:")
    print(f"   • {corr_matrix.shape[0]}x{corr_matrix.shape[1]} sucursales")
    print(f"   • Correlaciones entre ventas de sucursales")
    print(f"   • Lista para heatmap")
    
    print(f"\n🎯 Este notebook usará datos REALES para heatmaps")
    
    USAR_DATOS_REALES = True
    
except Exception as e:
    print(f"\n⚠️  No se pudo cargar la tabla de Unity Catalog")
    print(f"   Error: {e}")
    print(f"\n📝 Solución:")
    print(f"   1. Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
    print(f"   2. Verifica que la tabla exista: {CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    print(f"\n   Continuando con datos sintéticos...")
    
    df_ventas = None
    corr_matrix = None
    USAR_DATOS_REALES = False

print("\n" + "="*70)

## 📚 Heatmaps: Visualización de Matrices

### 🔥 ¿Qué es un Heatmap?

Un **heatmap** (mapa de calor) es una visualización donde los valores de una matriz se representan con **colores**.

**Concepto:**
* Valor alto = Color caliente (🔴 Rojo)
* Valor bajo = Color frío (🔵 Azul)

---

### 📊 Caso 1: Matriz de Correlación

**Correlación:** Mide la relación lineal entre dos variables (-1 a +1).

**Matriz de correlación:**
```
         Ventas  Costos  Gastos
Ventas    1.00    0.85   -0.30
Costos    0.85    1.00   -0.15
Gastos   -0.30   -0.15    1.00
```

**Interpretación:**
* 🔴 **+1.0:** Correlación perfecta positiva
* ⚪ **0.0:** Sin correlación
* 🔵 **-1.0:** Correlación perfecta negativa

**En el heatmap:**
* Ventas y Costos: 🔴 Rojo intenso (0.85) → Fuerte correlación
* Ventas y Gastos: 🔵 Azul (-0.30) → Correlación negativa

---

### ⚠️ Caso 2: Matriz de Riesgo

**Matriz de Riesgo:** Probabilidad vs Impacto

```
              Insignificante  Menor  Moderado  Mayor  Crítico
Muy probable      🟡          🟠     🟠       🔴     🔴
Probable          🟢          🟡     🟠       🟠     🔴
Posible           🟢          🟢     🟡       🟠     🟠
Improbable        🟢          🟢     🟢       🟡     🟠
Raro              🟢          🟢     🟢       🟢     🟡
```

**Uso:** Priorizar riesgos empresariales.

---

### 🎨 Escalas de Color

**Escalas comunes:**

| Escala | Uso típico |
|--------|---------------|
| **RdYlGn** (Rojo-Amarillo-Verde) | Indicadores de performance |
| **Blues** (Azules) | Densidad, frecuencia |
| **RdBu** (Rojo-Azul) | Correlaciones (positivo/negativo) |
| **Viridis** | Científica (perceptualmente uniforme) |

---

### 🛠️ Construcción en Plotly

**Heatmap básico:**
```python
import plotly.express as px

fig = px.imshow(
    corr_matrix,
    labels=dict(color="Correlación"),
    color_continuous_scale='RdBu_r',  # Escala Rojo-Azul invertida
    zmin=-1, zmax=1  # Rango de correlación
)
fig.show()
```

**Con anotaciones:**
```python
import plotly.graph_objects as go

fig = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.index,
    text=corr_matrix.values.round(2),  # Texto en celdas
    texttemplate="%{text}",
    colorscale='RdBu_r'
))
fig.show()
```

---

### 💼 Casos de Uso Empresariales

1. **Finanzas:** Correlación entre activos (portafolio)
2. **Ventas:** Patrones por región y producto
3. **RRHH:** Performance por departamento y mes
4. **Riesgo:** Matriz probabilidad-impacto
5. **Logística:** Tiempos de entrega por ruta y fecha

In [0]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

print("🔥 HEATMAPS: CORRELACIÓN Y RIESGO")
print("="*70)

print(f"\nVersión de Pandas: {pd.__version__}")
print(f"Versión de NumPy: {np.__version__}")
try:
    import plotly
    print(f"Versión de Plotly: {plotly.__version__}")
except:
    print("⚠️  Plotly no instalado. Ejecuta: %pip install plotly")

print("\n🎯 En este notebook aprenderás:")
print("  • px.imshow() - Heatmaps rápidos")
print("  • go.Heatmap() - Heatmaps personalizados")
print("  • Matrices de correlación")
print("  • Matrices de riesgo")

print("\n📖 Métodos clave:")
print("  - df.corr()  # Calcular correlaciones")
print("  - px.imshow(matrix, color_continuous_scale='RdBu_r')")
print("  - go.Heatmap(z=values, x=cols, y=rows)")

print("\n" + "="*70)
print("✅ Librerías cargadas correctamente")

In [0]:
# 🎯 OPCIONAL: Calcular correlaciones desde datos reales

# Descomentar para usar datos reales:
"""
import pandas as pd
import plotly.express as px

print("💾 Cargando datos reales desde Unity Catalog...")

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    df_ventas = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3").toPandas()
    df_ventas['fecha'] = pd.to_datetime(df_ventas['fecha'])
    
    # Crear matriz de ventas (filas=fechas, columnas=sucursales)
    df_pivot = df_ventas.pivot_table(
        index='fecha', 
        columns='sucursal_id', 
        values='ventas'
    )
    
    # Calcular matriz de correlación entre sucursales
    df_corr = df_pivot.corr()
    
    print(f"✅ Datos reales cargados: {len(df_ventas):,} registros")
    print(f"   Sucursales: {df_ventas['sucursal_id'].nunique()}")
    print(f"\n📊 Matriz de correlación calculada:")
    display(df_corr)
    
    print(f"\n💡 Variables disponibles:")
    print("   • df_corr: Matriz de correlación entre sucursales")
    print("   • df_pivot: Ventas pivotadas (fechas x sucursales)")
    print("\n   Ejemplo de heatmap con Plotly:")
    print("   fig = px.imshow(df_corr, text_auto=True, aspect='auto')")
    
except Exception as e:
    print(f"⚠️  Tabla no encontrada: {e}")
    print("   Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
"""

print("ℹ️  Usando datos sintéticos de este notebook")
print("="*70)

In [0]:
# Código de inicialización de notebook reindexado
import pandas as pd
import numpy as np
print('Notebook reindexado listo para práctica en Databricks')